# Course 1: Lidar
## Part 4: Working with Real PCD
#### By Jonathan L. Moran (jonathan.moran107@gmail.com)
From the Sensor Fusion Nanodegree programme offered at Udacity.

## Objectives

## 1. Introduction

### 1.1. The "City Block" Scene

In this lesson, we make use of a new simulated driving environment: the "City Block" scene. In this scene are multiple obstacles captured by a real LiDAR sensor mounted to an actual "self-driving car". The scene itself captures a four-way intersection with moderate traffic density with a diverse set of obstacles occupying the "City Block" scene, e.g., cars, road signs, buildings, etc.

### 1.2 Filtering with Point Cloud Library (PCL)

Individual point cloud "scans", depending on the LiDAR sensor specifications, contain hundreds of thousands of data points. Each of these scans are recorded at very high intervals, typically between 10-30 frames per second. In order to process each one of these scans in "real-time", many system engineers choose to [downsample](https://en.wikipedia.org/wiki/Downsampling_(signal_processing)) the LiDAR point clouds. By intentionally filtering / removing many of the LiDAR points in each scan, we can "speed up" processing time and drastically reduce the file sizes of the point clouds. The two techniques for downsampling we will discuss here are _grid voxelisation_ and _region-based filtering_. 

[Voxelisation](https://pointclouds.org/documentation/tutorials/voxel_grid.html) is the process of fitting 3D geometry onto the captured scene in order to perform point manipulation inside each fitted region. For example, we can "represent" the flat surface of a table as a simple 3D "box". Using the 3D shape to define the object dimensions, we are able to constrain the points scanned off the table's surface then reduce the points in the "voxel" area using e.g., centroid approximation, which preserves only one centre point inside each "leaf" formed by the voxel grid. Here, a "leaf" refers to a pre-defined area (e.g., $1 cm$) in which the voxel grid is discretised (i.e., divided up). An example of downsampling using the Point Cloud Library (PCL) [`pcl::VoxelGrid`](https://pointclouds.org/documentation/classpcl_1_1_voxel_grid_3_01pcl_1_1_p_c_l_point_cloud2_01_4.html#ad3efe8cb07386b291c88c5178e8b135d) filter is visualised in [this YouTube video](https://www.youtube.com/watch?v=YHR6_OIxtFI&t) where the original scan (left) is reduced (right) with a centroid approximation-based leaf method. This is a type of uniformly-distributed voxelisation method — when strategically configured — allows the shape of the objects captured to be relatively preserved. However, this method may not be practical as it requires "foresight" into the object geometry being scanned to strategically determine the appropriate "leaf" size(s) that will result in a desirable point density while preserving the underlying object representation. While there are more-advanced techniques for voxelisation (e.g., deep learning-based [PointNet](https://github.com/charlesq34/pointnet), [PointNet++](https://stanford.edu/~rqi/pointnet2/) and [VoxelNet](https://arxiv.org/abs/1711.06396)), we will leave the exploration into those methods for you to consider.

[Region-based filtering]() is another category of techniques for reducing the number of total points in a LiDAR scan. With region-based filtering, we have the ability to "apply" multiple geometric representations across sub-regions of a single unstructured point cloud. Using the statistical multi-scale interest region-based extraction [1] method, we can employ a data-driven approach to selecting the location(s) of our "interest regions" amd the "support radius" value(s) they are defined by. Unlike with the uniformly-distributed voxel grid method above, this mulit-scale region-based method allows us to "break apart" the point cloud into sub-regions, each with varying "geometries". The multi-scale interest region method allows us to e.g., reduce redundancy in areas with little shape variation, and better represent sparsity in these local "neighbourhoods" of points, by guiding the selection process with locally-informed features (called "descriptors") within the sub-regions. We can use the Unnikrishnan et al. (2008) method natively in the Point Cloud Library (PCL) by creating a [`pcl::StatisticalMultiscaleInterestRegionExtraction`](https://pointclouds.org/documentation/classpcl_1_1_statistical_multiscale_interest_region_extraction.html) class instance.

### 1.3 Streaming with Point Cloud Library (PCL) 

In the previous Exercises `E1.4.0` through `E1.4.2(b)`, we created functions to process and filter indivdiual `.pcd` files, one file at a time. In this section, we will create an overloaded `cityBlock()` function which works on multiple `.pcd` files. Our goal is to "stream" the input point cloud data, frame by frame, and use the overloaded `cityBlock()` function to filter each frame. In simplistic terms, we want to create a "filtering pipeline" which handles more than one input frame.

## 2. Programming Task

### The "City Block" Scene

#### E1.4.0: `cityBlock()`

In this section we create the `cityBlock()` function inside [`environment.cpp`]() which performs the following steps:
1. Create a point processor instance that stores `pcl::PointXYZI` data;
2. Load the PCD file and render its data onto the PCL Viewer.


##### The `cityBlock()` function

```cpp
// From J. Moran's `src/environment.cpp`:
// Credit: 
```

```cpp
// In `src/environment.cpp`:

void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    /** E1.4.0: Render the `CityBlock` Scene. **/
    // Creating a new point processor (stores Intensity values)
    ProcessPointClouds<
        pcl::PointXYZI
    > *pointProcessorI = new ProcessPointClouds<pcl::PointXYZI>();
    // Loading the `CityBlock` point cloud data
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr inputCloud = pointProcessorI->loadPcd(
        "../src/sensors/data/pcd/data_1/0000000000.pcd"
    );
    // Rendering point cloud data onto PCL Viewer canvas
    renderPointCloud(
        viewer,
        inputCloud,
        "inputCloud — City Block Scan"
    );
}
```

##### Testing the `cityBlock()` function

With the `cityBlock()` function defined, we will now call it from inside the `main()` programme function instead of the previous `simpleHighway()` scene. 

```cpp
// In `src/environment.cpp`:

int main() {
    // ..
    /** E1.1.0: Create 3D highway scene. **/
    // simpleHighway(viewer);
    /** E1.4.0: Render the `CityBlock` Scene. **/
    cityBlock(viewer);
}
```

To run the `cityBlock()` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

With the point cloud data (PCD) file loaded from its directory path (`"../src/sensors/data/pcd/data_1/0000000000.pcd"`), we obtain the following output:

```console
starting enviroment
Loaded 119978 data points from ../src/sensors/data/pcd/data_1/0000000000.pcd
```

<img src="figures/2024-07-06-Figure-1-cityBlock-Rendering.png" alt="Figure 1. The cityBlock scene — Rendered with Point Cloud Library (PCL)." height="70%" width="70%">

$$\textrm{Figure 1. The cityBlock scene — Rendered with Point Cloud Library (PCL).}$$

#### E1.4.1: `FilterCloud()` with `pcl::VoxelGrid()`

##### The `pcl::VoxelGrid()` method

```cpp
// From J. Moran's `src/processPointClouds.cpp`:
// Credit:
```

```cpp
// In `src/processPointClouds.cpp`:

template<typename PointT> typename pcl::PointCloud<
    PointT
>::Ptr ProcessPointClouds<PointT>::FilterCloud(
    typename pcl::PointCloud<PointT>::Ptr cloud, 
    float filterRes, 
    Eigen::Vector4f minPoint, 
    Eigen::Vector4f maxPoint
) {
    // Time segmentation process
    auto startTime = std::chrono::steady_clock::now();
    /** E1.4.1: Filtering the point cloud. **/
    // TODO:: Fill in the function to do voxel grid point reduction and region based filtering
    typename pcl::PointCloud<PointT>::Ptr cloudFiltered(
        new pcl::PointCloud<PointT>
    );
    // Creating the voxel-based filtering object
    pcl::VoxelGrid<PointT> vg;
    vg.setInputCloud(cloud);
    // Specifying the leaf size / "cell" dimensions
    vg.setLeafSize(filterRes, filterRes, filterRes);
    vg.filter(*cloudFiltered);
    auto endTime = std::chrono::steady_clock::now();
    auto elapsedTime = std::chrono::duration_cast<
        std::chrono::milliseconds
    >(endTime - startTime);
    std::cout << "filtering took "
              << elapsedTime.count() << " milliseconds\n";
    return cloudFiltered;
}
```

##### Testing the `pcl::VoxelGrid()` method

To apply the filtering function, call the `FilterCloud()` function we defined above with the `pointProcessorI` instance from inside the `cityBlock()` function, as follows:

```cpp
// In `src/environment.cpp`:

void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    // ..
    /** E1.4.1: Filtering with `pcl::VoxelGrid` **/
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr filterCloud = pointProcessorI->FilterCloud(
        inputCloud,
        0.2f,
        Eigen::Vector4f(0.0, 0.0, 0.0, 1.0), // minPoint; ignore for now.
        Eigen::Vector4f(0.0, 0.0, 0.0, 1.0) // maxPoint; ignore for now.
    );
    std::cerr << "Loaded " << filterCloud->points.size()
              << " data points from filtered cloud\n";
    // ..
}
```

To run the `cityBlock()` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

To view the filtered point cloud, you must pass the `filterCloud` instance into the `renderPointCloud()` function on `line 108` inside the `cityBlock()` function:

```cpp
void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    // ..
    // Rendering point cloud data onto PCL Viewer canvas
    renderPointCloud(
        viewer,
        filterCloud, // inputCloud; replace to view filtered cloud instead
        "inputCloud — City Block Scan (filtered)"
    );
}
```

With the _voxel grid filtered_ point cloud data (PCD) file loaded from its directory path (`"../src/sensors/data/pcd/data_1/0000000000.pcd"`), we obtain the following output:

```console
starting enviroment
Loaded 119978 data points from ../src/sensors/data/pcd/data_1/0000000000.pcd
filtering took 7 milliseconds
Loaded 23273 data points from filtered cloud
```

<img src="figures/2024-07-06-Figure-2-cityBlock-Rendering-Voxel-Grid-Filtering-Comparison.png" alt="Figure 2. The `cityBlock` scene — Unmodified (left) versus Filtered (right) point cloud downsampled with Voxel Grid method using Point Cloud Library (PCL)." height="70%" width="70%">

$$\textrm{Figure 2. The `cityBlock` scene — Unmodified (left) versus Filtered (right) point cloud downsampled with Voxel Grid method using Point Cloud Library (PCL).}$$

To produce the above downsampled point cloud, a voxel "cell" dimension of `0.2f` was provided (the input argument `filterRes` to the `FilterCloud()` function).

#### E1.4.2(a): `FilterCloud` with `pcl::CropBox` ("scene")

In this section we "downsample" our given point cloud scan by selectively removing points that fall within a defined region.

For our test case, we want to utilise `pcl::CropBox()` to define two regions of interest: the first, the "scene", will be the total point cloud "area" we wish to preserve — all points _outside_ this defined region will be eliminated. The other area of interest, the ego-vehicle "roof", will be the area blanketing the roof of the ego-vehicle — all points _inside_ this defined region will be eliminated.

Let's start with the first region, the "scene". Similar to the above E1.4.1, we make use of the `FilterCloud()` method to _filter_ (eliminate) points in the cloud which lie _outside_ the defined region of interest. To do so, we pass in non-zero values for the `minPoint` and `maxPoint` input arguments. These two `Eigen::Vector4f` coordinate pairs define the rectangular area of the total point cloud area we wish to preserve.

##### The `pcl::CropBox()` method ("scene")

```cpp
// From J. Moran's `src/processPointClouds.cpp`:
// Credit:
```

```cpp
// In `src/processPointClouds.cpp`:

template<typename PointT> typename pcl::PointCloud<
    PointT
>::Ptr ProcessPointClouds<PointT>::FilterCloud(
    typename pcl::PointCloud<PointT>::Ptr cloud, 
    float filterRes, 
    Eigen::Vector4f minPoint, 
    Eigen::Vector4f maxPoint
) {
    // ..
    /** E1.4.2(a): Filtering the point cloud with `pcl::CropBox`. **/
    typename pcl::PointCloud<PointT>::Ptr cloudRegion(new pcl::PointCloud<PointT>);
    // Defining the first region: the area of points to preserve
    pcl::CropBox<PointT> regionPreserved(true);
    regionPreserved.setMin(minPoint);
    regionPreserved.setMax(maxPoint);
    regionPreserved.setInputCloud(cloudFiltered);
    // Cropping the point cloud to the desired region (the "scene")
    regionPreserved.filter(*cloudRegion);
    // ..
}
```

##### Testing the `pcl::CropBox()` method ("scene")

In order to make use of the region-based cropping method, we must assign the two input arguments `minPoint` and `maxPoint` values associated with the region we wish to "crop". These two variables are defined as `Eigen::Vector4f` types, where each represents a 3D point.

For our test case, we want to utilise [`pcl::CropBox()`]() to define the first region of interest as the total area of the point cloud which we are interested in. We hand-select these values such that all anticipated scene details remain preserved, while eliminating points which are either "too far" away from the ego-vehicle to be useful for processing, or points which are unlikely to provide significant information to act upon (e.g., points belonging to adjacent buildings or traffic from multiple lanes away). 

Side note: this is just for demonstration purposes, and not design advice for actual AVs. We leave that for you to reason about. A few techniques that may be used are histograms (to plot $x$-, $y$-, and $z$-coordinate value distributions and determine "range of interest"), and repeated parameter tuning of the `minPoint` and `maxPoint` values based on the desired range one wishes to acheive for the given scene being considered.



```cpp
// From J. Moran's `src/environment.cpp`:
// Credit:
```

```cpp
// In `src/environment.cpp`:

void cityBlock{
    // ..
    /** E1.4.2(a): Filtering the point cloud with `pcl::CropBox`. **/
    // NOTE: choosing non-zero valued vectors for `minPoint`, `maxPoint`;
    // These define the area of the region we wish to preserve. 
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr regionCloud = pointProcessorI->FilterCloud(
        inputCloud,
        0.2f,
        Eigen::Vector4f(-15.0, -6.0, -3.0, 1),
        Eigen::Vector4f(30.0, 6.0, 10.0, 1)
    );
    // ..
}
```

Note that the above values for the `minPoint` and `maxPoint` were selected based off of a previous students' findings (linked [here](https://github.com/tortillafun/_udacity_/blob/main/src/environment.cpp); credit: [@tortillafun](https://github.com/tortillafun)). However, in [this Knowledge forum post](https://knowledge.udacity.com/questions/1044231) course mentor Marcelo advises we determine these values "pragmatically" through a combination of visualization, empirical observation, and manual tuning (their advice replicated below):

>**Visualization**:
>* Visualize the point cloud and manually inspect the bounds of the car. Identify the coordinates that encompass the car and some buffer space around it.
>
>**Empirical Observation**:
>* Observe the point cloud data to see the distribution of points. Note the approximate coordinates that define the extents of the car.
>
>**Manual Tuning**:
>* Start with an initial guess for minPoint and maxPoint. Adjust these values iteratively based on the results of the filtering. Ensure that all relevant points (including the car and any immediate surroundings) are included.

To make sense of this, we can implement a "point-picking event" with callback function ([example here](https://stackoverflow.com/questions/26699427/checking-point-coordinates-in-pclvisualizer)), or perform a trial-and-error process of iteratively revising the "random guess" of these values until a satisfactory result is obtained. For now, none of these two approaches seem to be conclusive enough to proceed with. Luckily, for this scene (`../data/pcd/data_1/0000000000.pcd`), we seem to have found a "good enough" set of values to preserve a majority of the scene objects while rejecting all other points (i.e., the building "walls").

To run the `cityBlock()` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

To view the region-based filtered point cloud, you must pass the `regionCloud` instance into the `renderPointCloud()` function on `line 124` inside the `cityBlock()` function:

```cpp
void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer
) {
    // ..
    // Rendering point cloud data onto PCL Viewer canvas
    renderPointCloud(
        viewer,
        regionCloud, // Alternatives: `inputCloud` or `filterCloud`
        "regionCloud — City Block Scan (filtered with region-based method)"
    );
    // ..
}
```

The _region-based filtered_ point cloud (`regionCloud`) contains data (in `.pcd` format) from the file loaded at the directory path (`"../src/sensors/data/pcd/data_1/0000000000.pcd"`). The following console output is obtained:

```console
starting enviroment
Loaded 119978 data points from ../src/sensors/data/pcd/data_1/0000000000.pcd
filtering took 7 milliseconds
filtering took 32 milliseconds
Loaded 6771 data points from filtered cloud
```

<img src="figures/2024-07-06-Figure-3-cityBlock-Rendering-Region-Based-Filtering-Scene-Preserved-Comparison.png" alt="Figure 3. The 'cityBlock' scene — Filtered (Voxel-based; left) versus Filtered (Voxel- and Region-based; right) point cloud downsampled with both Voxel- and Region-Based Filtering with Point Cloud Library (PCL).">

$$\textrm{Figure 3. The 'cityBlock' scene — Filtered (Voxel-based; left) versus Filtered (Region-based; right) point cloud downsampled with Region-Based Filtering with Point Cloud Library (PCL).}$$

#### E1.4.2(b): `FilterCloud` with `pcl::CropBox` ("roof")

In this second `pcl::CropBox()` task, we wish to filter the _second_ region of interest: the "roof" of the ego-vehicle — all points _inside_ this defined region will be eliminated.

Just as before, we must define a `minPoint` and `maxPoint` vector of values which effectively outline the area we wish to consider. In contrast to the previous task, the area we define here will be used to _remove_ all points _inside_ the region, rather than _outside_, as done before.

In `E1.4.2(b)`, we define these vector values in the `FilterCloud()` function from directly:

##### The `pcl::CropBox()` method ("scene")

```cpp
// From J. Moran's `src/processPointClouds.cpp`:
// Credit:
```

```cpp
// In `src/processPointClouds.cpp`:

template<typename PointT> typename pcl::PointCloud<
    PointT
>::Ptr ProcessPointClouds<PointT>::FilterCloud(
    typename pcl::PointCloud<PointT>::Ptr cloud, 
    float filterRes, 
    Eigen::Vector4f minPoint, 
    Eigen::Vector4f maxPoint
) {
    // ..
    /** E1.4.2(b): Filtering the "scene" with `pcl::CropBox`. **/
    // Creating a vector to store the indices determined to belong to the "roof"
    std::vector<int> indicesRoof;
    pcl::CropBox<PointT> roof(true);
    // Defining the points which form the area of the roof to filter out
    // NOTE: choosing non-zero valued vectors for `minPoint`, `maxPoint`;
    // These define the area of the region we wish to eliminate. 
    roof.setMin(
        Eigen::Vector4f(-1.5, -1.7, -1.0, 1)
    );
    roof.setMax(
        Eigen::Vector4f(2.6, 1.7, -0.4, 1)
    );
    roof.setInputCloud(cloudRegion);
    roof.filter(indicesRoof);
    // Populating the data structure with indices of the roof 
    pcl::PointIndices::Ptr inliers{new pcl::PointIndices};
    for (int i = 0; i < indicesRoof.size(); i++) {
        inliers->indices.push_back(indicesRoof[i]);
    };
    // Extracting the roof indices (i.e., deleting them from point cloud)
    pcl::ExtractIndices<PointT> extract;
    extract.setInputCloud(cloudRegion);
    extract.setIndices(inliers);
    extract.setNegative(true);
    extract.filter(*cloudRegion);
    // ..
}
```

##### Testing the `pcl::CropBox()` method ("roof")

Using the values provided by Udacity for the `minPoint` and `maxPoint` (as defined in the `FilterCloud()` function), we test the region-based cropping method with the following steps.

First, call the `FilterCloud()` function with the input arguments from `E1.4.2(a)` (that is, the `minPoint` and `maxPoint` previously selected such that the _preserved_ area is defined inside these coordinates).

```cpp
// From J. Moran's `src/environment.cpp`:
// Credit:
```

```cpp
// In `src/environment.cpp`:

void cityBlock{
    // ..
    /** E1.4.2(a)-E1.4.2(b): Filtering the point cloud with `pcl::CropBox`. **/
    // NOTE: choosing non-zero valued vectors for `minPoint`, `maxPoint`;
    // These define the area of the region we wish to preserve. 
    // CANDO: Modify these values to select a different area to preserve points within.
    pcl::PointCloud<
        pcl::PointXYZI
    >::Ptr regionCloud = pointProcessorI->FilterCloud(
        inputCloud,
        0.2f,
        Eigen::Vector4f(-15.0, -6.0, -3.0, 1),
        Eigen::Vector4f(30.0, 6.0, 10.0, 1)
    );
    // ..
}
```

To test other `minPoint` and `maxPoint` values used to define the region we _eliminate_ points inside (the task of `E1.4.2(b)`, then modify the lines `87-88` of `processPointClouds.cpp` directly). In other words, pass the values as `Eigen::Vector4f` arguments to the `roof.setMin( .. )` and `roof.setMax( .. )` function calls, respectively. 

To run the `cityBlock()` scene and render the LiDAR point scan in the PCL Viewer, use the following build commands in a console window: 
```console
root@foobar:/../1-1-Lidar-Obstacle-Detection/#  mkdir build && cd build
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  cmake ..
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  make
root@foobar:/../1-1-Lidar-Obstacle-Detection/build/#  ./environment
```

<img src="figures/2024-07-06-Figure-4-cityBlock-Rendering-Region-Based-Filtering-Scene-Preserved-Comparison-Scaled.png" alt="Figure 4. The 'cityBlock' scene — Filtered (Voxel- and Region-based with roof; left) versus Filtered (Voxel- and Region-based without roof; right) point cloud downsampled with both Voxel- and Region-Based Filtering with Point Cloud Library (PCL).">

$$\textrm{Figure 4. The 'cityBlock' scene — Filtered (Voxel- and Region-based } \textit{with roof} \textrm{; left) versus Filtered (Voxel- and Region-based } \textit{with roof} \textrm{; left) point cloud downsampled with Region-Based Filtering with Point Cloud Library (PCL).}$$

NOTE: the point cloud shown in Figure 4 above has been scaled (zoomed in) for clarity. 

#### E1.4.3: Streaming with `cityBlock()`

This overloaded `cityBlock()` function in `E1.4.3` will be designed to handle "multiple" `.pcd` files, one loaded per function call. The key difference between this overloaded function and the original `cityBlock()` function from `E1.4.0` is the additional input arguments, which allow the overloaded `cityBlock()` function to be called _without_ needing to instantiate the `pointProcessorI` instance with each call (i.e., to avoid re-creating this object at every frame). Also, since the point cloud input will vary frame-to-frame, the `inputCloudI` is now an input argument (specified optionally as a `const` type declaration, since we do not modify it inside the function itself). With this overloaded `cityBlock()` function, we no longer need to create the point processor instance nor load the point cloud file from _inside_ the function itself, as performed in the original `cityBlock()` function.

##### The overloaded `cityBlock()` function

```cpp
// From J. Moran's `src/environment.cpp`:
// Credit:
```

```cpp
// In `src/environment.cpp`: 

void cityBlock(
    pcl::visualization::PCLVisualizer::Ptr &viewer, 
    ProcessPointClouds<pcl::PointXYZI> *pointProcessorI, 
    const pcl::PointCloud<pcl::PointXYZI>::Ptr &inputCloudI
) {
    /** E1.4.3: File stremaing with overloaded `cityBlock()`. **/
    // ..
}
```

##### Testing the overloaded `cityBlock()` function

In order to make use of the overloaded `cityBlock()` function, we modify the `main()` programme function to now loop over all `.pcd` files contained inside a specified folder (this folder path is initialised below as the `stream` iterator).

```cpp
// From J. Moran's `src/environment.cpp`:
// Credit:
```

```cpp
// In `src/environment.cpp`: 

int main(
    int argc,
    char **argv
) {
    // ..
    /** E1.4.0: Render the `CityBlock` Scene. **/
    // NOTE: The previous `cityBlock` call is "commented out"
    //cityBlock(viewer);
    /** E1.4.3: File streaming with overloaded `cityBlock()`. **/
    // Now defining the point processor outside `environment::cityBlock`
    ProcessPointClouds<
        pcl::PointXYZI
    > *pointProcessorI = new ProcessPointClouds<
        pcl::PointXYZI
    >();
    // Creating list of all `.pcd` files to "stream"
    // CANDO: Modify folder pointing to `.pcd` file(s)
    std::vector<
        boost:filesystem::path
    > stream = pointProcessorI->streamPcd(
        "../src/sensors/data/pcd/data_1"
    );
    // Creating file path "iterator"
    auto streamIterator = stream.begin();
    pcl:PointCloud<pcl::PointXYZI>::Ptr inputCloudI;
    while (!viewer->wasStopped()) {
        /** E1.4.3: File streaming with overloaded `cityBlock()`. **/
        // Clearing the PCL Viewer canvas of any previous elements
        viewer->removeAllPointClouds();
        viewer->removeAllShapes();
        // Loading current `.pcd` file
        inputCloudI = pointProcessorI->loadPcd(
            (*streamIterator).string()
        );
        // Performing obstacle detection process on current `.pcd` file
        // Calling the overloaded `cityBlock` "streaming" function
        cityBlock(
            viewer,
            pointProcessorI,
            inputCloudI
        );
        // Advancing the file iterator to the next `.pcd` file
        streamIterator++;
        if (streamIterator == stream.end()) {
            // Looping to first file in folder
            streamIterator = stream.begin();
        }
        // Refresh PCL Viewer canvas (with default time-step of 1ms)
        viewer->spinOnce(
            1
        );
    }
    // ..
}

The modified `main` programme function above loads each `.pcd` file found in the given folder sequentially, processes each `.pcd` scan in the overloaded `cityBlock()` function, then renders its contents (the "processed" point cloud and any obstacles detected) onto the PCL Viewer canvas. The rate at which each "frame" (a single `.pcd` file) is processed inside the `while (!viewer->wasStopped())` loop is governed by the `viewer->spinOnce()` call inside the loop; if you wish to control this timing, specify an input [`time`](http://pointclouds.org/documentation/classpcl_1_1visualization_1_1_p_c_l_visualizer.html#a896556f91ba6b45b12a9543a2b397193) argument (in milliseconds) which will effectively set a "refresh rate" at which new `.pcd` files are fetched and processed from the folder.

## 3. Closing Remarks

#### Alternatives
* `E1.4.0`: Render a different scene (choose from the available `.pcd` files inside the [`"../src/sensors/data/pcd/data_1/"`]() sub-directory);
* `E1.4.1`: Experiment with different voxel "cell" dimensions (i.e., choose a different value for the `filterRes` input arugument and observe its impact on the number of points in the filtered cloud as compared to the original);
* `E1.4.2(a)`: Experiment with different `minPoint` / `maxPoint` values to determine a desirable region of interest to filter (preserve);
* `E1.4.2(b)`: Experiment with different `minPoint` / `maxPoint` values to determine a desirable region of interest to filter (eliminate);
* `E1.4.3`: Select a different "folder" in which to fetch and process `.pcd` files with the overloaded `cityBlock` function (i.e., modify the file path given as an input argument to the `stream` vector on `line 377`);
* `E1.4.3`: Modify the "refresh rate" of the overloaded `cityBlock()` pipeline (i.e., specify a `time` input argument to the `viewer->spinOnce()` function call on `line 405`);

#### Extensions of task
* `E1.4.2(a)`: Use various tools (e.g., histogram, min-max finder, [pick-point callback function](https://stackoverflow.com/questions/26699427/checking-point-coordinates-in-pclvisualizer) etc.) to determine more-optimal `minPoint` and `maxPoint` values such that the majority of the desired point cloud is preserved after clipping (and such that "important" data is not filtered out);
* `E1.4.3`: 


## 4. Future Work

* ⬜️
* ✅

## Credits

This assignment was prepared by Aaron Brown and Michael Maile of Mercedes-Benz Research & Development of North America (MBRDNA), 2021 (link [here](https://learn.udacity.com/nanodegrees/nd313/)).


References
* [] Unnikrishnan, R., Hebert, M. Multi-scale interest regions from unorganized point clouds. Workshop on Search in 3D (S3D), IEEE Conference on Computer Vision and Pattern Recognition (CVPR). June 2008. [doi:10.1109/CVPRW.2008.4563030](http://dx.doi.org/10.1109/CVPRW.2008.4563030).


Helpful resources:
* 